# The Duckiedrone ToF data flow

[Notebook 1](./1-tof-basics.ipynb) followed one sensor and one number. On a Duckiedrone that
  number does not reach a program directly. It passes through several programs first, and each
  of them can change it. This notebook follows that chain from the chip to the program written
  in [Notebook 3](./3-subscribing-to-tof.ipynb).

Programs on a robot rarely hand data to each other directly. Instead, one program announces its
  data on a named channel, and any program that wants that data asks to be given a copy whenever
  a new data point appears. Neither side needs to know the other exists, so a program can be
  started, stopped, or rewritten without disturbing the rest of the robot. A system of named
  channels like this is often called **middleware**, because it sits between the programs that
  produce data and the programs that consume it.

The Duckiedrone uses the Robot Operating System, [ROS 2](https://docs.ros.org/), whose
  terminology is:

- data channel: **topic**,

- putting data on it: **publishing**,

- asking for copies: **subscribing**,

- a program that takes part in this exchange of data: **node**.

For more on ROS, see the [Duckiebot ROS learning
  experience](https://docs.duckietown.com/ente/duckietown-manual/60-learning-experiences/lx-setup-ros-basics.html).

<!-- TODO: Replace with DD ROS2 LX -->

The range readings arrive on a topic, and the program written in Notebook 3 is a node that
  subscribes to it. ROS 2 carries the reading only from the bridge onwards: the hop before it
  and the hop to the flight controller use other protocols, each explained at its step. The same
  programs run on a virtual Duckiedrone, where the readings are simulated instead of measured.

By the end of the notebook you should be able to answer:

- Which pieces of software handle a range reading between the chip and a subscriber?

- What does each one add to, or remove from, the reading?

- What sets how often a new reading arrives?

- Why do a target the sensor cannot detect and a target beyond 3.6 m arrive on the topic as the
  same value?

- Why does a node that subscribes to the range and then crashes not affect the flight
  controller?

## The big picture

This notebook follows the downward sensor: it runs whenever the Duckiedrone is on, and it is the
  one the flight controller uses. The diagram shows the rates it is configured for.

```
              I2C                        DTPS                       ROS 2
+----------+ 30 Hz +-------------------+ 30 Hz +------------------+ 30 Hz +--------------+
| VL53L1X  |------>| ToF driver        |------>| ROS 2 bridge     |------>| Node from    |
| (bottom) |       | driver-tof-bottom |       | ros2-tof-bottom  |       | Notebook 3   |
+----------+       +-------------------+       +------------------+       +--------------+
                                                        |
                                                        | ROS 2, 30 Hz
                                                        v
                                           +------------------------+
                                           | MAVROS2                |
                                           | ros2-mavros            |
                                           | distance_sensor plugin |
                                           +------------------------+
                                                        |
                                                        | MAVLink DISTANCE_SENSOR, 30 Hz
                                                        v
                                           +------------------------+
                                           | Flight controller      |
                                           | PX4, EKF2              |
                                           +------------------------+
```

Each box between the sensor and the flight controller is one program. Each program runs in its
  own
  [**container**](https://docs.docker.com/get-started/docker-concepts/the-basics/what-is-a-container/),
  an isolated package holding the program and everything it needs to run, under the name shown
  in its box. On a physical Duckiedrone these containers run on the onboard computer, a
  Raspberry Pi. On a virtual Duckiedrone they run on the computer hosting it, and the flight
  controller is simulated. The chain branches after the bridge: the same readings that reach the
  node from Notebook 3 also reach the flight controller.

<!-- TODO: point the container link at the Duckiedrone Docker LX once it exists. -->

## Step 1. From the VL53L1X to the Raspberry Pi

<a id="step-1"></a>

Each [VL53L1X](https://www.st.com/en/imaging-and-photonics-solutions/vl53l1x.html) connects to
  the Raspberry Pi over [**I2C**](https://en.wikipedia.org/wiki/I%C2%B2C), a two-wire bus on
  which the computer addresses each device by a number. The chip does all of the work described
  in [Notebook 1](./1-tof-basics.ipynb) on its own: it fires the pulses, builds the histogram,
  finds the peak, and holds the result, in millimeters, until the Raspberry Pi reads it.

Every VL53L1X starts up at the same I2C address, `0x29`. The address can be changed, but only
  while that sensor is the one answering at `0x29`, and the change is lost when power is
  removed, so sharing one bus would mean bringing the five sensors up one at a time on every
  boot. The Duckiedrone's sensor board avoids that with an I2C multiplexer, a
  [TCA9548A](https://www.ti.com/product/TCA9548A), which gives each sensor a channel of its own.
  Linux presents each channel to the Raspberry Pi as a separate bus:

| Sensor | Port | I2C bus | Points |
|---|---|---|---|
| `bottom` | CHL0 | 23 | Straight down |
| `left` | CHL1 | 24 | To the Duckiedrone's left |
| `right` | CHL2 | 25 | To the Duckiedrone's right |
| `top` | CHL3 | 26 | Straight up |
| `front` | CHL4 | 27 | Forward |

The bus numbers are the ones a Raspberry Pi 4 assigns to the multiplexer's channels. Left and
  right are named from the Duckiedrone's point of view, and its front is the side the camera
  faces.

## Step 2. The driver

<a id="step-2"></a>

One container per sensor runs the program that reads the chip, named `driver-tof-bottom`,
  `driver-tof-front`, and so on. At startup it opens the sensor's bus and programs the settings
  from [Notebook 1](./1-tof-basics.ipynb): the distance mode, the timing budget, and the
  inter-measurement period. It also announces the sensor's fixed properties once: its field of
  view, and the minimum and maximum distance it is specified for in that mode.

Then it repeats the same steps for every measurement:

1. **It waits for a new measurement.** The chip signals when a measurement is ready. Without
   that check, the driver would read the previous result again and publish it as if it were new.

2. **It converts to meters.** The chip reports millimeters, while ROS and the rest of the
   Duckietown software work in meters.

3. **It drops unusable readings.** Two cases leave the driver with no distance to report: the
   chip found no peak that cleared the noise, or the distance is beyond the mode's maximum, 3.6
   m in `LONG_RANGE`. In both cases the driver publishes a reading with no value.

4. **It stamps the reading** with the time it read the measurement.

5. **It publishes on DTPS.** Duckietown moves messages between its own programs with the
   [Duckietown Postal Service](https://docs.duckietown.com/ente/duckietown-manual/04-software-tools/duckietown-postal-service-dtps.html)
   (DTPS). The driver publishes at the path `sensor/time_of_flight/<sensor_name>/range`.

On a virtual Duckiedrone this step is where the difference lies. There is no chip and no I2C
  bus, so the
  [Duckiematrix](https://docs.duckietown.com/ente/duckietown-manual/50-duckiematrix/getting-started/duckiematrix-first-steps.html)
  connects to the same DTPS path and supplies readings computed by the simulator. The
  Duckiematrix has to be running, with the Duckiedrone attached to an entity, that is, linked to
  a drone in the simulated world; otherwise the driver is running but has nothing to publish.

## Step 3. The ROS 2 bridge

<a id="step-3"></a>

A second container per sensor, `ros2-tof-bottom` for the downward one, receives each DTPS
  message and republishes it as a standard ROS 2 message of type `sensor_msgs/Range`, on the
  topic:

```
/<robot>/bottom_tof_driver_node/range
```

where `<robot>` is the Duckiedrone's hostname.

The `range` field has to hold a number, so the bridge fills a reading with no value with
  `999.0`. It does the same for any distance above `max_range`. The value is not arbitrary. The
  [message
  definition](https://github.com/ros2/common_interfaces/blob/jazzy/sensor_msgs/msg/Range.msg)
  says of `range`: "values < range_min or > range_max should be discarded" (the fields
  themselves are named `min_range` and `max_range`), so a number far outside those limits is the
  message's own way of marking a reading to discard.

This is also why the two cases from [Step 2](#step-2) look the same on the topic: a target the
  chip cannot detect and a target beyond 3.6 m both arrive as `999.0`. Applying the limits is
  left to whatever reads the topic. The node written in Notebook 3 prints raw values, so that
  its output shows readings on both sides of them.

### What the bridge fills in

A `Range` message holds more than a distance. The bridge takes the sensor's fixed properties
  from the driver's announcement and repeats them in every message:

| Field | What arrives | Note |
|---|---|---|
| `header.stamp` | When the driver read the measurement | Carried through from the driver |
| `header.frame_id` | `<robot>/tof/<sensor_name>` | Set by the driver |
| `range` | Meters, or `999.0` | The measurement |
| `min_range` | `0.04` | The closest distance the sensor is specified for, not a lower limit on what it reports |
| `max_range` | `3.6` | The driver's maximum in `LONG_RANGE` |
| `field_of_view` | `0.471` | Radians, the 27 degree cone |
| `radiation_type` | `1` | The message definition's code for `INFRARED` |
| `variance` | `0.0` | The message reads `0` as "variance unknown", not as "no noise" |

The chip does not timestamp its readings, so the driver stamps each one when it reads it. The
  bridge keeps that stamp instead of using the time the message reached it, so `header.stamp` is
  when the driver read the distance, as the message definition asks.

[Notebook 3](./3-subscribing-to-tof.ipynb) prints one of these messages from a Duckiedrone, to
  check against this table.

## Step 4. The other consumer: the flight controller

<a id="step-4"></a>

The flight controller runs [PX4](https://docs.px4.io/main/en/) and holds the Duckiedrone level
  and at height, so it needs an estimate of its height above the floor. The downward ToF sensor
  does not measure that height, as [Notebook 1](./1-tof-basics.ipynb) showed: it measures the
  distance along its own axis to whatever surface is below. With the Duckiedrone level over a
  bare floor, the two are the same, so this distance is the closest the Duckiedrone gets to a
  height measurement. PX4 combines its sensor readings into an estimate of the Duckiedrone's
  state, height included, with an estimator called
  [EKF2](https://docs.px4.io/main/en/advanced_config/tuning_the_ecl_ekf.html), and on the
  Duckiedrone EKF2 is configured to take its height from this distance.

The flight controller is not a ROS 2 node. It speaks [MAVLink](https://mavlink.io/en/), a
  compact message protocol for drones, over a USB cable on a physical Duckiedrone and a UDP
  socket on a virtual one. The `ros2-mavros` container translates between the two: it runs
  [MAVROS2](https://github.com/mavlink/mavros/tree/ros2), a ROS 2 node that relays messages
  between ROS 2 and MAVLink. Its `distance_sensor` plugin subscribes to the range topic like any
  other node and forwards each reading to PX4 as a MAVLink `DISTANCE_SENSOR` message, marked as
  facing down.

Each subscriber gets its own copy of every message, so a crashing subscriber node does not
  affect the flight controller. If the driver or the bridge stops, every subscriber loses the
  distance, the flight controller included.

## Step 5. From the bridge to a subscriber

<a id="step-5"></a>

The last stop is a node that subscribes to `/<robot>/bottom_tof_driver_node/range`. It receives
  the `Range` message described in [Step 3](#step-3), with the conversion to meters, the `999.0`
  substitution, and the timestamp already applied. What it does with each message is up to that
  node, and [Notebook 3](./3-subscribing-to-tof.ipynb) writes such a node.

## Not all five sensors are treated the same

| Sensor | Configured rate (Hz) | Running by default |
|---|---|---|
| `bottom` | 30 | Yes |
| `front` | 5 | No |
| `left` | 5 | No |
| `right` | 5 | No |
| `top` | 5 | No |

The five sensors differ in two ways. The downward sensor is configured for 30 Hz and starts with
  the Duckiedrone, while the other four are configured for 5 Hz and stay off until someone
  starts them.

Both differences save processor time on the Raspberry Pi, which the camera and the flight
  software also need. Each sensor runs two programs, a driver and a bridge, and most of their
  work is repeated for every message. Measured on one physical Duckiedrone, a driver and its
  bridge together used about 12% of one processor core at 30 Hz, and about 2% at 5 Hz.

The flight controller needs the downward distance at all times, so that sensor gets the higher
  rate and runs from boot. The other four are started by hand, when they are needed.

### Turning the other four on

Only `driver-tof-bottom` and `ros2-tof-bottom` start with the Duckiedrone. The eight containers
  for the other four sensors belong to a separate stack, `duckiedrone/extra_sensors`, which
  starts only on request, so until then only one range topic exists. [Notebook
  4](./4-all-five-sensors.ipynb) starts it and reads all five sensors.

## Changing any of this

Each driver reads its settings from a file on the Duckiedrone:

```
/data/config/node/tof_driver_<sensor_name>/<robot>.yaml
```

If the file does not exist when the driver starts, the driver creates it from its defaults. From
  then on the driver reads only this file, so editing it changes the settings, and a later
  update to the defaults does not overwrite those edits. The downward sensor's default file,
  with some comments and entries removed:

```yaml
sensor_name: "bottom"
sensor_model: "VL53L1X"

# publish rate, Hz
frequency: 30

# SHORT_RANGE: less reach, better in bright light. LONG_RANGE: more reach.
mode: "LONG_RANGE"

# photon collection time per measurement, ms: 15 (SHORT_RANGE only), 20, 33,
# 50, 100, 200, or 500. Longer is less noisy and reaches further, but slower.
timing_budget: 33

# time between measurements, ms, at least timing_budget. Set from frequency if omitted.
# inter_measurement_period: 33

display_fragment_frequency: 0
connectors:
  # Raspberry Pi 4, port CHL0
  -   bus: 23
      address: 0x29
```

`mode`, `timing_budget`, and `inter_measurement_period` are the chip settings from [Notebook
  1](./1-tof-basics.ipynb). `frequency` belongs to the driver, but it also sets the chip's pace:
  when `inter_measurement_period` is left out, the driver sets it to $1 / \text{frequency}$, or
  to the timing budget if that is longer.

The chip cannot finish measurements faster than one per timing budget, or one per
  inter-measurement period if that is longer. If `frequency` asks for more than the chip can
  deliver, the driver logs a warning and publishes at the highest rate the chip allows. It does
  not repeat a measurement to make up the difference, because of the check in [Step 2](#step-2).

Two mistakes stop the driver at startup with an error: a timing budget the chip does not accept,
  and an `inter_measurement_period` shorter than the timing budget.

The other four sensors use `frequency: 5` with `timing_budget: 200`, the longest budget that
  fits in 1/5 s. To run one of them faster, shorten its budget too, for example `frequency: 10`
  with `timing_budget: 100`.

On a physical Duckiedrone the file belongs to root, so edit it with `sudo`. Both the edit and
  the restart happen on the Duckiedrone, after logging in as in [Notebook
  3](./3-subscribing-to-tof.ipynb), Step 1. After editing, restart the sensor's two containers,
  the driver first and then the bridge:

```bash
docker restart driver-tof-bottom
docker restart ros2-tof-bottom
```

The order matters. The bridge subscribes to the driver's DTPS channel once, when it starts, and
  never subscribes again, so a bridge left running across a driver restart stops publishing.